# NovaRetail · Privacidad, triaje y evaluación de riesgos (P3, P4)
**Persona 4 · Entrega 1 · Semilla 42**

Analizo los datos personales presentes en soporte y evalúo un triaje por reglas.
El tratamiento del texto se realiza en memoria: **no se muestran ni guardan tickets sin enmascarar**.
Para reproducirlo, ejecutar primero `uv sync` y `uv run novaretail all`.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from novaretail import privacy

DATA_DIR = Path(os.getenv("NOVARETAIL_DATA_DIR", ROOT / "data"))
RAW, META = DATA_DIR / "raw", DATA_DIR / "_meta"
FIGURES = ROOT / "docs" / "figuras"
FIGURES.mkdir(parents=True, exist_ok=True)
assert (RAW / "tickets.json").exists(), "Primero ejecuta uv run novaretail all"

with open(RAW / "tickets.json", encoding="utf-8") as f:
    tickets = pd.DataFrame(json.load(f))
customers = pd.read_csv(RAW / "customers.csv", dtype=str)
orders = pd.read_csv(RAW / "orders.csv", dtype=str)
labels = pd.read_csv(META / "tickets_labels.csv", dtype=str)
print(f"Tickets: {len(tickets):,} · Etiquetas de referencia: {len(labels):,}")

## 1. Inventario y finalidad de los datos
El inventario incluye clientes, identificadores vinculables en pedidos y datos de proveedor de facturas. Las bases jurídicas y la conservación propuestas requieren validación formal del DPO. No son plazos impuestos universalmente por ley.

In [ ]:
inv = privacy.personal_data_inventory({"customers": customers, "tickets": tickets, "orders": orders})
print(f"Campos identificados: {len(inv)}")
display(inv[["conjunto", "campo", "categoria", "finalidad", "base_legal", "conservacion", "acceso"]])

## 2. Enmascarado y auditoría de fuga observable
El texto se enmascara antes de clasificarlo. Los datos de clientes/pedidos se usan solo en esta fase para localizar el nombre completo asociado al ticket. La auditoría inspecciona valores sintéticos conocidos y direcciones con código postal. El **0 % observable no equivale a anonimización garantizada**: las erratas, valores sin formato reconocible y los identificadores de pedido siguen requiriendo controles.

In [ ]:
masked = privacy.mask_tickets(tickets, customers, orders)
audit = privacy.audit_masking(tickets, masked, customers, orders)
print("Ejemplo de salida (solo texto ya enmascarado):")
display(masked[["ticket_id", "channel", "text_masked"]].head(4))
display(audit)
print("Fuga observable global:", f"{100*audit.sin_enmascarar.sum()/max(audit.detectadas_original.sum(),1):.2f}%")

## 3. Clasificador por reglas y comparación con las etiquetas del generador
Las etiquetas de control (`data/_meta`) se usan exclusivamente para **evaluar**, nunca para asignar la categoría. La categoría se obtiene solo del texto enmascarado; el segmento del cliente se incorpora después para el análisis de equidad.

In [ ]:
pred = privacy.classify_tickets(masked)
pred = privacy.link_ticket_segments(pred, orders, customers)
results = privacy.evaluate_classification(pred, labels)
print(f"Tickets evaluados: {results['n']:,}")
print(f"Exactitud categoría: {results['acierto_categoria_pct']:.2f}%")
print(f"Exactitud prioridad: {results['acierto_prioridad_pct']:.2f}%")
print(f"Recall prioridad alta: {results['recall_alta_pct']:.2f}%")
display(results["por_categoria"])
display(results["por_grupo"])

## 4. Diferencias entre canales y segmentos
La comparación es descriptiva y se hace sobre datos sintéticos. Las diferencias entre grupos no demuestran discriminación causal. Las categorías y la prioridad no usan como entrada el segmento, pero las etiquetas del generador sí contemplan el segmento «premium» para algunas prioridades; esa diferencia debe interpretarse como una decisión de política comercial a revisar.

In [ ]:
groups = results["por_grupo"]
for field, image_name, metric, heading in [
    ("channel", "p4_error_categoria_canal.png", "error_categoria_pct", "Error de categoría por canal"),
    ("segmento", "p4_error_prioridad_segmento.png", "error_prioridad_pct", "Error de prioridad por segmento"),
]:
    d = groups[groups["dimension"].eq(field)].copy()
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(d["grupo"], d[metric])
    ax.set_title(heading)
    ax.set_ylabel("Porcentaje de tickets (%)")
    ax.set_ylim(0, max(10, d[metric].max()*1.18))
    ax.tick_params(axis="x", rotation=20)
    for i, value in enumerate(d[metric]):
        ax.text(i, value+0.35, f"{value:.1f}%", ha="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES / image_name, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
print("Cobertura de vinculación de segmentos:", f"{100*pred.segmento.ne('sin_vincular').mean():.1f}%")

## 5. Evidencia resumida para la memoria y decisión al DPO

**Hallazgos y controles prioritarios:**

1. La fuga observable debe mantenerse en cero para los patrones auditados, con revisión humana de casos dudosos; no se debe equiparar una regex con anonimización.
2. Los falsos negativos de prioridad alta exigen una cola de revisión humana de tickets no clasificados y una segunda revisión de incidencias sensibles.
3. La diferencia de error por segmento requiere revisar el criterio de prioridad «premium» que se usa en el generador: atender mejor un segmento puede desplazar injustamente casos de otros.
4. El DPO debe aprobar conservación, base jurídica, autorización por roles y transferencias de la futura plataforma cloud.
5. Ante un incidente de seguridad, activar protocolo de contención, análisis y, cuando proceda, notificación conforme al RGPD.

**MAGERIT:** la matriz de amenazas e impacto, con salvaguardas y riesgo residual, se documenta en el apartado 5.6 del Word. La valoración presentada es una estimación del caso, no una auditoría operativa de una empresa real.

In [ ]:
print("RESUMEN PARA EL WORD")
print("Dataset sintético: ", results["n"], "tickets")
print("Fuga observable de PII auditada:",
      f"{100*audit.sin_enmascarar.sum()/max(audit.detectadas_original.sum(),1):.2f}%",
      "(alcance limitado a patrones y valores observables)")
print("Acierto categoría:", results["acierto_categoria_pct"], "%")
print("Acierto prioridad:", results["acierto_prioridad_pct"], "%")
print("Recall alta:", results["recall_alta_pct"], "%")
print("Los resultados deben copiarse de esta ejecución final, con uv.lock, al Word.")